In [1]:
from openai import OpenAI
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
import gradio as gr


from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

### Load Documents

In [3]:
loader = DirectoryLoader(
    "./documents",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(len(documents))

3


### Split Documents

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

7


### Create Embeddings

In [5]:
embedding_model = OpenAIEmbeddings(
    openai_api_key="OPENROUTER_API_KEY",
    openai_api_base="OPENROUTER_API_BASE_URL",
)

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Store in Vector Database

In [7]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./vectordb"
)

### Query the Database

In [8]:
query = "What is the leave policy?"

results = vector_db.similarity_search(query, k=3)

results

[Document(id='31e9fb27-198b-4d75-93ca-bfe92d28431c', metadata={'source': 'documents/policy.txt'}, page_content='Company Leave Policy\n\nEmployees are entitled to 20 days of paid annual leave each year.\nSick leave allows employees to take up to 10 days per year with a medical certificate.\nMaternity leave is granted for 12 weeks.\nEmployees must submit leave requests at least 5 days in advance through the HR portal.'),
 Document(id='2d6647c3-6a16-4f57-b702-0f686c9e8b39', metadata={'source': 'documents/policy.txt'}, page_content='Company Leave Policy\n\nEmployees are entitled to 20 days of paid annual leave each year.\nSick leave allows employees to take up to 10 days per year with a medical certificate.\nMaternity leave is granted for 12 weeks.\nEmployees must submit leave requests at least 5 days in advance through the HR portal.'),
 Document(id='d1b89c1d-4cd0-4e1a-8855-3fda5baf6fde', metadata={'source': 'documents/fag.txt'}, page_content='7. Where can I find company policies and guid

In [9]:
def ask_question(question):

    # Retrieve relevant documents
    results = vector_db.similarity_search(question, k=3)

    # Combine context
    context = "\n".join([doc.page_content for doc in results])

    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )

    answer = response.choices[0].message.content

    return answer

In [10]:
interface = gr.Interface(
    fn=ask_question,
    inputs=gr.Textbox(label="Ask a question"),
    outputs=gr.Textbox(label="Answer"),
    title="📚 HR Policy Assistant",
    description="This chatbot answers questions based on company documents using Retrieval-Augmented Generation (RAG).",
    theme="soft"
)

/Users/sirphiltechpaxe/projects/llm_lab/.venv/lib/python3.12/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [16]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv
